# ¿Es el bitcoin el nuevo oro?

**Proyecto EDA – Módulo Data Analysis**  
Autor: Joaquín Flores Serradilla  
Entrega 1: tema, hipótesis y datasets

## 1. Contexto y pregunta

Un **activo refugio** es aquel que mantiene su valor, o incluso sube, cuando los mercados caen. El referente clásico es el **oro**, y la **renta fija** (bonos de EE. UU.) también se considera refugio.

En los últimos años se ha extendido la idea de que el bitcoin es el "oro digital". Este proyecto analiza con datos si esa idea se sostiene.

**Pregunta:** ¿se comporta el bitcoin como un activo refugio, es decir, protege el capital cuando la bolsa cae?

## 2. Hipótesis

**H1 – Crisis.** En las crisis bursátiles, el bitcoin cae junto con la bolsa en lugar de protegerse.

**H2 – Comparativa con refugios.** En esas crisis, el oro y la renta fija protegen el capital y el bitcoin no.

**H3 – Tipos de interés.** El bitcoin cae en las fases de subida de tipos de la Reserva Federal y sube en las de bajada, igual que la bolsa.

**H4 – Halving y volatilidad.** El bitcoin sigue ciclos ligados al halving (sube con fuerza en los meses posteriores) y su volatilidad se ha reducido de un ciclo a otro.

## 3. Datasets

| Dato | Fuente | Frecuencia | Desde |
|---|---|---|---|
| Bitcoin (BTC/USD) | Investing.com | Diaria | 01/11/2013 |
| S&P 500 (renta variable) | Investing.com | Diaria | 2010 |
| Oro (XAU/USD) | Investing.com | Diaria | 2010 |
| Bonos de EE. UU. a largo plazo (TLT, renta fija) | Investing.com | Diaria | 2010 |
| Tipos de interés de la Reserva Federal (FEDFUNDS) | FRED (Reserva Federal de San Luis) | Mensual | 2010 |

**Periodo de análisis:** noviembre de 2013 – actualidad (marcado por el inicio de los datos de bitcoin).

**Uso de cada dato:**
- **S&P 500:** identifica las crisis (días y periodos de caída de la bolsa).
- **Oro y bonos:** refugios con los que se compara el bitcoin.
- **Tipos de la Fed:** clasifican los periodos en fases de subida, bajada o tipos estables.

**Crisis previstas para el análisis:** crash bursátil chino (agosto 2015), Brexit (junio 2016), caídas de 2018, COVID (marzo 2020), inflación y subida de tipos (2022) y crisis bancaria de EE. UU. (marzo 2023).

**Halvings dentro del periodo:** julio 2016, mayo 2020 y abril 2024.

## 4. Limitaciones conocidas

- Los datos de bitcoin empiezan en noviembre de 2013, por lo que quedan fuera la crisis de Chipre (2013) y el halving de 2012.
- Solo hay tres ciclos de halving completos o en curso: las conclusiones de H4 serán orientativas.
- El bitcoin cotiza todos los días; la bolsa, el oro y los bonos solo en días laborables. Para comparar se usarán solo los días con todos los mercados abiertos.
- Los tipos de la Fed son mensuales, frente a los precios diarios.
- Que dos variables se muevan a la vez no demuestra que una cause la otra.

## 5. Primera carga de los datos

Los CSV de Investing.com vienen con formato español (punto en los miles y coma en los decimales) y fechas `dd.mm.aaaa`. En cada archivo:
1. Se lee el CSV indicando ese formato numérico.
2. Se convierte la fecha de texto a fecha.
3. Se conserva solo la fecha y el precio de cierre (`Último`).
4. Se ordena de más antiguo a más reciente y la fecha pasa a ser el índice.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Bitcoin
df_btc = pd.read_csv("../data/Datos históricos del Bitcoin.csv", thousands=".", decimal=",")
df_btc["Fecha"] = pd.to_datetime(df_btc["Fecha"], format="%d.%m.%Y")
df_btc = df_btc[["Fecha", "Último"]].rename(columns={"Último": "bitcoin"})
df_btc = df_btc.sort_values("Fecha").set_index("Fecha")
df_btc.info()

In [ ]:
# S&P 500
df_sp500 = pd.read_csv("../data/Datos históricos del S&P 500.csv", thousands=".", decimal=",")
df_sp500["Fecha"] = pd.to_datetime(df_sp500["Fecha"], format="%d.%m.%Y")
df_sp500 = df_sp500[["Fecha", "Último"]].rename(columns={"Último": "sp500"})
df_sp500 = df_sp500.sort_values("Fecha").set_index("Fecha")
df_sp500.info()

In [ ]:
# Oro
df_oro = pd.read_csv("../data/Datos históricos XAU_USD.csv", thousands=".", decimal=",")
df_oro["Fecha"] = pd.to_datetime(df_oro["Fecha"], format="%d.%m.%Y")
df_oro = df_oro[["Fecha", "Último"]].rename(columns={"Último": "oro"})
df_oro = df_oro.sort_values("Fecha").set_index("Fecha")
df_oro.info()

In [ ]:
# Bonos de EE. UU. (TLT)
# Cambia el nombre del archivo por el que tenga tu CSV descargado de Investing.com
df_bonos = pd.read_csv("../data/Datos históricos TLT.csv", thousands=".", decimal=",")
df_bonos["Fecha"] = pd.to_datetime(df_bonos["Fecha"], format="%d.%m.%Y")
df_bonos = df_bonos[["Fecha", "Último"]].rename(columns={"Último": "bonos"})
df_bonos = df_bonos.sort_values("Fecha").set_index("Fecha")
df_bonos.info()

In [ ]:
# Tipos de interés de la Reserva Federal (mensual)
df_fed = pd.read_csv("../data/FEDFUNDS.csv", parse_dates=["observation_date"])
df_fed = df_fed.rename(columns={"observation_date": "Fecha", "FEDFUNDS": "tipos_fed"}).set_index("Fecha")
df_fed.info()

## 6. Primer vistazo

Para comparar activos con precios muy distintos (el bitcoin vale decenas de miles de dólares y el oro unos miles), se lleva cada serie a **base 100** en la primera fecha común. Así se ve cuánto ha crecido cada uno desde el mismo punto de partida.

Se usa escala logarítmica en el eje vertical porque el crecimiento del bitcoin es tan grande que, en escala normal, el resto de líneas quedarían planas.

In [ ]:
# Unir solo los días en que cotizan todos los activos
precios = df_btc.join([df_sp500, df_oro, df_bonos], how="inner")

# Base 100 en la primera fecha común
base100 = precios / precios.iloc[0] * 100

base100.plot(figsize=(12, 6), logy=True, title="Evolución base 100 (escala logarítmica)")
plt.ylabel("Valor (base 100)")
plt.show()

## 7. Próximos pasos

1. Completar la limpieza y unir todas las series por fecha.
2. Calcular rentabilidades diarias, volatilidad y caídas máximas.
3. Resolver cada hipótesis con estadísticos y gráficos.
4. Redactar conclusiones y preparar la presentación final (20 de noviembre).